In [1]:
pip install langchain langchain-community langchain-openai fastembed faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 9.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 48.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 48.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.6/607.6 kB 27.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

load_dotenv()  # .env 파일 로드

import os

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


# ---------------------------------------------------------
# 1) DeepSeek: R1(free) 모델 연결
# ---------------------------------------------------------
llm = ChatOpenAI(
    # ⚠ 여기만 DeepSeek R1 → 다른 모델로 교체
    # 예시: deepseek/deepseek-chat, gpt-4o-mini, mistral/mistral-small 등
    model="deepseek/deepseek-chat",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
)


# ---------------------------------------------------------
# 2) 내부 문서(LLM은 모르는 정보) 예시
# ---------------------------------------------------------
internal_doc = """
[KB은행 NPS 운영 매뉴얼] (2025.10 개정)

1. NPS 설문 응답 데이터는 최소 5년간 보관한다.
2. NPS 설문 로그 및 원시 VOC 데이터는 2년간 보관 후,
   비식별화하여 추가 3년간 보관한다.
3. 고객 식별정보(이름, 주민등록번호 등)는 암호화 상태로 분리 저장한다.
"""

question = "KB은행 NPS 설문 로그 데이터는 몇 년 동안 보관해?"


# ---------------------------------------------------------
# 3) LLM 단독 (비-RAG)
# ---------------------------------------------------------
print("=== 1) LLM만 사용 (비 RAG) ===")
llm_only_answer = llm.invoke(
    f"""
너는 KB은행 직원이라고 가정하고 답해줘.

질문: "{question}"

주의:
- KB은행 내부 매뉴얼 내용을 실제로는 모르는 상태라고 가정해.
- 모르면 '정확히 알 수 없습니다.'라고 말해.
"""
)
print(llm_only_answer.content)

=== 1) LLM만 사용 (비 RAG) ===
정확히 알 수 없습니다.  

KB은행의 NPS 설문 로그 데이터 보관 기간은 내부 정책에 따라 다를 수 있으며, 해당 정보는 공개되지 않았습니다. 보다 정확한 내용은 KB은행 고객센터(1588-9999) 또는 담당 부서에 문의하시는 것이 좋습니다.  

도움이 되었으면 합니다! 😊


In [5]:
# ---------- B. 내부 문서 + RAG 파이프라인 ----------
# 1. 임베딩 & 벡터스토어 생성 (FastEmbed = 로컬 임베딩, 무료)
embedding = FastEmbedEmbeddings()
vectorstore = FAISS.from_texts(
    texts=[internal_doc],
    embedding=embedding,
    metadatas=[{"source": "KB_NPS_manual_2025_10"}],
)
retriever = vectorstore.as_retriever(k=3)


# 2. RAG용 프롬프트 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 KB은행 NPS 운영 매뉴얼을 바탕으로만 답하는 어시스턴트이다.

- 반드시 아래 '관련 문서' 내용 안에서만 근거를 찾아 대답한다.
- 문서에 없는 내용은 추측하지 말고
  '매뉴얼에 근거가 없어 정확히 알 수 없습니다.'라고 답한다.
- 답변 끝에 근거가 된 문장을 그대로 한 줄 정도 인용해줘.
""",
        ),
        (
            "human",
            "질문: {question}\n\n[관련 문서]\n{context}",
        ),
    ]
)


# 3. 검색된 문서를 보기 좋게 이어붙이는 함수
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


# 4. LCEL 기반 RAG 체인 구성
rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | format_docs,
    }
    | prompt
    | llm
)

print("\n=== 2) RAG 사용 (문서 검색 + 답변 생성) ===")
rag_answer = rag_chain.invoke(question)
print(rag_answer.content)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:02<00:00,  2.24it/s]



=== 2) RAG 사용 (문서 검색 + 답변 생성) ===
KB은행 NPS 설문 로그 데이터는 2년간 보관 후, 비식별화하여 추가 3년간 보관합니다. 

근거: "NPS 설문 로그 및 원시 VOC 데이터는 2년간 보관 후, 비식별화하여 추가 3년간 보관한다." (KB은행 NPS 운영 매뉴얼 2025.10 개정)


---

# 🔍 LLM 단독 vs RAG 비교 데모

이 데모는 다음 3가지를 **직관적으로 체감**하게 만드는 것을 목표로 합니다.

1. **현실 1: LLM은 우리 조직의 최신/내부 문서를 모른다.**
2. **현실 2: 근거 없는 답변과, 문서 기반 정확한 답변의 차이.**
3. **현실 3: LLM + Retrieval + Workflow(RAG 체인)의 필요성.**

---

## 🧪 1) LLM 단독 사용 사례

질문  
> “KB은행은 고객 로그나 VOC 데이터를 얼마나 보관하나요?”

LLM 단독 답변 예시:
- “KB은행은 보통 3년~5년 정도 보관합니다.”  
- 또는 “내부 정책을 알 수 없기 때문에 정확한 보관 기간을 말씀드리기 어렵습니다.”

**특징**
- 내부 정책을 모르기 때문에 **그럴듯한 추측**(hallucination) 또는  
  **모른다고 답변**하는 두 가지 패턴이 모두 나타날 수 있음.
- 최신 문서·규정·내부 정책은 **파라미터에 들어 있지 않음**.

---

## 🧪 2) RAG 사용 사례

동일 질문  
> “KB은행은 고객 로그나 VOC 데이터를 얼마나 보관하나요?”

RAG 파이프라인이 내부 문서를 검색한 뒤:

내부 매뉴얼 문장 예:
- “로그 및 원시 VOC 데이터는 2년 보관 후 비식별화하여 추가 3년 보관한다.”

RAG 답변 예시:
- “로그 데이터는 **2년 보관 후 비식별화하여 추가 3년 보관**,  
  총 **5년**입니다.  
  (근거: ‘로그 및 원시 VOC 데이터는 2년 보관 후 비식별화하여 추가 3년 보관한다’)”

**특징**
- **정확한 수치** + **근거 문장**을 함께 제시  
- 최신 정책 변경도 문서만 업데이트하면 **즉시 반영**

---
